# Notebook 01 — Data Loading & Cleaning
**Dataset:** DataCo Supply Chain (Kaggle, ~180k transaksi)  
**Catatan:** File CSV menggunakan encoding `latin-1`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
OUT = Path('../output')
FIGURES = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

## 1. Load Data

In [2]:
df = pd.read_csv(RAW / 'DataCoSupplyChainDataset.csv', encoding='latin-1')
print(f'Shape: {df.shape}')
df.head(2)

Shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class


## 2. Rename Kolom (bersihkan spasi & karakter khusus)

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('/', '_', regex=False)
)
print('Kolom setelah rename:')
print(df.columns.tolist())

Kolom setelah rename:
['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shippin

## 3. Parse Tanggal & Feature Engineering

In [4]:
# Cari kolom tanggal
date_cols = [c for c in df.columns if 'date' in c.lower()]
print('Kolom tanggal:', date_cols)

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Identifikasi kolom lead time
ship_real = [c for c in df.columns if 'shipping' in c and 'real' in c]
ship_sched = [c for c in df.columns if 'shipment' in c and 'scheduled' in c]
print('Lead time real    :', ship_real)
print('Lead time sched   :', ship_sched)

Kolom tanggal: ['order_date_dateorders', 'shipping_date_dateorders']
Lead time real    : ['days_for_shipping_real']
Lead time sched   : ['days_for_shipment_scheduled']


In [5]:
# Nama kolom standar berdasarkan dataset DataCo
lead_real  = 'days_for_shipping_real'
lead_sched = 'days_for_shipment_scheduled'
deliv_col  = 'delivery_status'

# Fallback jika nama sedikit berbeda
if lead_real not in df.columns:
    lead_real = [c for c in df.columns if 'shipping' in c and 'real' in c][0]
if lead_sched not in df.columns:
    lead_sched = [c for c in df.columns if 'shipment' in c and 'scheduled' in c][0]

df['actual_lead_time']    = df[lead_real]
df['scheduled_lead_time'] = df[lead_sched]
df['delay_days']          = df['actual_lead_time'] - df['scheduled_lead_time']
df['is_late']             = df[deliv_col].str.contains('Late', case=False, na=False).astype(int)

# Bulan order
order_date_col = [c for c in df.columns if 'order_date' in c or 'dateorders' in c]
if order_date_col:
    df['order_year_month'] = df[order_date_col[0]].dt.to_period('M')

print('Feature engineering selesai.')
print(f"Late delivery rate: {df['is_late'].mean():.1%}")

Feature engineering selesai.
Late delivery rate: 54.8%


## 4. Drop Kolom Tidak Relevan

In [6]:
drop_keywords = ['image', 'description', 'latitude', 'longitude', 'zipcode', 'password']
drop_cols = [c for c in df.columns
             if any(kw in c.lower() for kw in drop_keywords)]
df = df.drop(columns=drop_cols)
print(f'Drop {len(drop_cols)} kolom: {drop_cols}')
print(f'Shape setelah drop: {df.shape}')

Drop 7 kolom: ['customer_password', 'customer_zipcode', 'latitude', 'longitude', 'order_zipcode', 'product_description', 'product_image']
Shape setelah drop: (180519, 51)


## 5. Cek Missing Values

In [7]:
missing = df.isnull().sum()
print('Kolom dengan missing values:')
print(missing[missing > 0].sort_values(ascending=False))

# Drop baris dengan missing di kolom kunci
key_cols = ['actual_lead_time', 'scheduled_lead_time', 'is_late']
df = df.dropna(subset=key_cols)
print(f'\nShape final: {df.shape}')

Kolom dengan missing values:
customer_lname    8
dtype: int64

Shape final: (180519, 51)


## 6. Ringkasan Dataset

In [8]:
print('=== RINGKASAN DATASET ===')
print(f'Total transaksi       : {len(df):,}')
print(f'Produk unik           : {df["product_name"].nunique():,}')
print(f'Kategori unik         : {df["category_name"].nunique():,}')
print(f'Market                : {df["market"].unique().tolist()}')
print(f'Shipping modes        : {df["shipping_mode"].unique().tolist()}')
print(f'Avg actual lead time  : {df["actual_lead_time"].mean():.1f} hari')
print(f'Late delivery rate    : {df["is_late"].mean():.1%}')

=== RINGKASAN DATASET ===
Total transaksi       : 180,519
Produk unik           : 118
Kategori unik         : 50
Market                : ['Pacific Asia', 'USCA', 'Africa', 'Europe', 'LATAM']
Shipping modes        : ['Standard Class', 'First Class', 'Second Class', 'Same Day']
Avg actual lead time  : 3.5 hari
Late delivery rate    : 54.8%


## 7. Simpan ke Parquet

In [9]:
df.to_parquet(OUT / 'df_clean.parquet', index=False)
print(f'Tersimpan: df_clean.parquet — {len(df):,} baris, {df.shape[1]} kolom')

Tersimpan: df_clean.parquet — 180,519 baris, 51 kolom
